# 04 - Retrieval

Build hybrid BM25 + dense retrieval (BGE-M3, Chroma) + cross-encoder rerank
(`ms-marco-MiniLM-L-6-v2`) — the committed default architecture. Uses the chunks
and embeddings already produced by `02_chunking`/`03_embedding` — no re-embedding
here.

With only 8 chunks, the Step 7 comparison is good enough to validate the pipeline
mechanics work, not to draw statistically meaningful conclusions — that needs the
formal Ragas eval in `07_eval` once the corpus and query set are bigger.

## Step 1: Setup

Imports, load `chunks.json`, `embeddings_bge_m3.npy`, `chunk_ids_bge_m3.json`, and
the sample queries (from `03_embedding`, extended if needed).

In [1]:
import json
from pathlib import Path

import numpy as np

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

embeddings = np.load(EMBEDDINGS_PATH)
chunk_ids = json.loads(CHUNK_IDS_PATH.read_text(encoding="utf-8"))

assert len(chunk_ids) == embeddings.shape[0]

# Rough, unverified sample queries for smoke-testing only -- not a formal eval set.
# Non-English translations have not been checked by a native speaker.
#
# Language coverage: same two anchor questions (overtime pay, salary timing)
# translated into more languages, so language is the only variable that changes.
# Category coverage: one new English question per newly-added corpus category
# (work-permit, medical, help), so topic and language aren't tested at once.
SAMPLE_QUERIES = [
    {"language": "en", "text": "How much overtime pay am I entitled to?"},
    {"language": "en", "text": "When must my employer pay my salary?"},
    {"language": "ms", "text": "Bilakah majikan saya perlu bayar gaji saya?"},
    {"language": "ta", "text": "எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?"},
    {"language": "my", "text": "ကျွန်တော် ဘယ်လောက် အချိန်ပိုခ ရထိုက်သလဲ"},
    {"language": "th", "text": "ฉันมีสิทธิ์ได้รับค่าล่วงเวลาเท่าไหร่"},
    {"language": "vi", "text": "Chủ sử dụng lao động của tôi phải trả lương khi nào?"},
    {"language": "en", "text": "Who pays repatriation costs when my Work Permit ends?"},
    {"language": "en", "text": "How much medical insurance must my employer provide?"},
    {"language": "en", "text": "How can I contact MOM?"},
]

len(chunks), embeddings.shape, len(SAMPLE_QUERIES)

(16, (16, 1024), 10)

## Step 2: Build Chroma collection

Index the chunks into a persistent Chroma collection using the already-computed
BGE-M3 embeddings (aligned via `chunk_ids_bge_m3.json`) — no re-embedding.

In [2]:
import chromadb

CHROMA_DIR = Path("../data/processed/chroma")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Rebuild fresh each run rather than appending -- collection.add() isn't idempotent,
# and re-running with a changed corpus (e.g. more/fewer source pages) would either
# error on duplicate IDs or leave stale entries from a previous corpus version.
if "bge_m3" in [c.name for c in client.list_collections()]:
    client.delete_collection(name="bge_m3")
collection = client.create_collection(name="bge_m3")

collection.add(
    ids=chunk_ids,
    embeddings=embeddings.tolist(),
    documents=[chunks_by_id[cid]["text"] for cid in chunk_ids],
    metadatas=[
        {
            "document_id": chunks_by_id[cid]["document_id"],
            "url": chunks_by_id[cid]["url"],
            "heading_path": chunks_by_id[cid]["heading_path"],
        }
        for cid in chunk_ids
    ],
)

collection.count()

16

## Step 3: Dense retrieval

A function that embeds a query with BGE-M3 and runs a similarity search against
the Chroma collection, returning the top-k chunks.

In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)


def dense_retrieve(query: str, top_k: int = 5) -> list[dict]:
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
    return [
        {"chunk_id": chunk_id, "distance": distance}
        for chunk_id, distance in zip(results["ids"][0], results["distances"][0])
    ]


dense_retrieve(SAMPLE_QUERIES[0]["text"])

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 194956.36it/s]


[{'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2',
  'distance': 0.4845885932445526},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-3',
  'distance': 0.6208765506744385},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-4',
  'distance': 0.7224515080451965},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-0',
  'distance': 0.7274839878082275},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-0',
  'distance': 0.7368616461753845}]

## Step 4: BM25 index + retrieval

Build a BM25 index over chunk texts (`rank_bm25`) and a function that returns the
top-k chunks for a query by lexical overlap.

In [4]:
import re

from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25_corpus = [tokenize(chunks_by_id[cid]["text"]) for cid in chunk_ids]
bm25 = BM25Okapi(bm25_corpus)


def bm25_retrieve(query: str, top_k: int = 5) -> list[dict]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"chunk_id": cid, "score": score} for cid, score in ranked]


bm25_retrieve(SAMPLE_QUERIES[0]["text"])

[{'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-4',
  'score': np.float64(4.177394252340064)},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1',
  'score': np.float64(3.8213917606992354)},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2',
  'score': np.float64(2.3712308566601954)},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-3',
  'score': np.float64(1.895118594155087)},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-0',
  'score': np.float64(1.6813869430995632)}]

## Step 5: Hybrid combination

Merge dense and BM25 rankings into a single ranked list (reciprocal rank fusion).

In [5]:
def reciprocal_rank_fusion(*ranked_lists: list[dict], k: int = 60) -> list[dict]:
    scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, item in enumerate(ranked, start=1):
            chunk_id = item["chunk_id"]
            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

    ranked_fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [{"chunk_id": chunk_id, "rrf_score": score} for chunk_id, score in ranked_fused]


def hybrid_retrieve(query: str, top_k: int = 5) -> list[dict]:
    dense_results = dense_retrieve(query, top_k=10)
    bm25_results = bm25_retrieve(query, top_k=10)
    fused = reciprocal_rank_fusion(dense_results, bm25_results)
    return fused[:top_k]


hybrid_retrieve(SAMPLE_QUERIES[0]["text"])

[{'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2',
  'rrf_score': 0.032266458495966696},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-4',
  'rrf_score': 0.032266458495966696},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-3',
  'rrf_score': 0.031754032258064516},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1',
  'rrf_score': 0.03128054740957967},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-0',
  'rrf_score': 0.031009615384615385}]

## Step 6: Cross-encoder rerank

Rerank the hybrid top-k with `ms-marco-MiniLM-L-6-v2` down to a final top-k
(CLAUDE.md's default; cross-lingual quality is an open question to watch for in
Step 7).

In [6]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANKER_MODEL_NAME)


def rerank(query: str, candidates: list[dict], top_k: int = 5) -> list[dict]:
    pairs = [(query, chunks_by_id[c["chunk_id"]]["text"]) for c in candidates]
    scores = reranker.predict(pairs)
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [{"chunk_id": c["chunk_id"], "rerank_score": float(score)} for c, score in reranked[:top_k]]


hybrid_candidates = hybrid_retrieve(SAMPLE_QUERIES[0]["text"], top_k=10)
rerank(SAMPLE_QUERIES[0]["text"], hybrid_candidates)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3642.58it/s]


[{'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2',
  'rerank_score': 4.070030689239502},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-3',
  'rerank_score': -0.5469765663146973},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-0',
  'rerank_score': -3.925567150115967},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1',
  'rerank_score': -3.9870991706848145},
 {'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-4',
  'rerank_score': -4.511022567749023}]

## Step 7: Compare

Run the sample queries through dense-only, BM25-only, hybrid, and hybrid+rerank
side by side. Watch specifically for: does BM25 add anything for non-English
queries, or does hybrid reduce to dense-only there? Does the (English-only)
reranker behave differently on cross-lingual pairs?

In [7]:
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}


def label(chunk_id: str) -> str:
    chunk = chunks_by_id[chunk_id]
    doc_slug = chunk["url"].rstrip("/").split("/")[-1]
    return f"{doc_slug} :: {chunk['heading_path'] or '(intro)'}"


def excerpt(chunk_id: str, length: int = 400) -> str:
    text = chunks_by_id[chunk_id]["text"].replace("\n", " ")
    return text[:length] + ("..." if len(text) > length else "")


def relevance(query_embedding: np.ndarray, chunk_id: str) -> float:
    chunk_embedding = embeddings[chunk_id_to_idx[chunk_id]]
    return float(
        np.dot(query_embedding, chunk_embedding)
        / (np.linalg.norm(query_embedding) * np.linalg.norm(chunk_embedding))
    )


for query in SAMPLE_QUERIES:
    print(f"\n{'=' * 80}\nQuery [{query['language']}]: {query['text']}\n{'=' * 80}")

    query_embedding = embedding_model.encode(query["text"], convert_to_numpy=True)

    dense_results = dense_retrieve(query["text"], top_k=3)
    bm25_results = bm25_retrieve(query["text"], top_k=3)
    hybrid_results = hybrid_retrieve(query["text"], top_k=3)
    reranked_results = rerank(query["text"], hybrid_retrieve(query["text"], top_k=10), top_k=3)

    for method_name, results in [
        ("Dense", dense_results),
        ("BM25", bm25_results),
        ("Hybrid RRF", hybrid_results),
        ("Hybrid + rerank", reranked_results),
    ]:
        print(f"\n{method_name} (top 3):")
        for r in results:
            rel = relevance(query_embedding, r["chunk_id"])
            print(f"  [{rel:.3f}] {label(r['chunk_id'])}\n    {excerpt(r['chunk_id'])}")


Query [en]: How much overtime pay am I entitled to?

Dense (top 3):
  [0.758] hours-of-work-overtime-and-rest-days :: Overtime pay
    Hours of work, overtime and rest day > Overtime pay  Overtime work is all work in excess of the normal hours of work (excluding breaks).  You can claim overtime if you are:  - A non-workman earning a monthly basic salary of $2,600 or less. - A workman earning a monthly basic salary of $4,500 or less.   The overtime rate payable for non-workmen is **capped at the salary level of $2,600, or an hourl...
  [0.690] hours-of-work-overtime-and-rest-days :: Maximum hours of work > Maximum hours of overtime
    Hours of work, overtime and rest day > Maximum hours of work > Maximum hours of overtime  An employee can only work **up to 72 overtime hours** in a month.  Employers can apply for an exemption if they require employees to work more than the 72 hours of overtime in a month.  **Note:** these work activities will not be granted exemption.  **Work on rest d